# 12.7 · 分词 / Tokenization (BPE)

> **课程定位 / Where this fits**
> 第 7 课，**Part 12**。前面 GPT/BERT 都假设文本已经是一串"token"——但**文本到底怎么切成 token？**
> Lesson 7, **Part 12**. GPT/BERT assumed text is a sequence of "tokens" — but **how is text actually split into tokens?**
>
> 分词是 LLM 流水线**第一步**，看似平凡却影响深远(词表大小、序列长度、能否处理新词/多语言/代码、甚至模型能力)。三种粒度各有硬伤：**词级**(词表爆炸 + 处理不了没见过的词)、**字符级**(序列太长、语义弱)。现代大模型都用**子词分词(subword)**，最经典的是 **BPE(字节对编码)**——从字符开始，不断把**最高频的相邻对**合并成新 token，自动在"常用词整体、罕见词拆片"之间取得平衡。本课**从零实现 BPE**(训练 + 编码),彻底搞懂 GPT 的分词器。
> Tokenization is the **first step** of the LLM pipeline — mundane-looking but consequential (vocab size, sequence length, handling new words/multilingual/code, even model capability). Three granularities each have flaws: **word-level** (vocab explosion + can't handle unseen words), **char-level** (too long, weak semantics). Modern LLMs use **subword tokenization**, classically **BPE (Byte-Pair Encoding)** — start from characters and repeatedly merge the **most frequent adjacent pair** into a new token, automatically balancing "common words whole, rare words split." We **implement BPE from scratch** (training + encoding) to fully understand GPT's tokenizer.
>
> 💼 **实战/面试视角**："为什么用子词 / BPE 算法 / 词表大小权衡 / token 数影响成本 / 中文与代码分词" 是 LLM 工程常考。
> 💼 **Practical/interview angle:** "why subwords / BPE algorithm / vocab-size trade-off / token count = cost / Chinese & code tokenization" — common LLM engineering.

> 📐 **符号约定 / Notation**
> - 子词(subword) —— 介于字符和词之间的片段(如 "token", "##ization") / subword piece
> - 合并(merge) —— 把两个相邻 token 合成一个新 token / merge two adjacent tokens

> 💡 **面试相关 / Interview-relevant**
> - "为什么不用词级/字符级、要用子词"（出镜率 ★★★★★）
> - "BPE 的训练过程"（★★★★★）
> - "BPE / WordPiece / SentencePiece 区别"（★★★★）
> - "词表大小如何权衡"（★★★★）
> - "token 数和 API 成本/上下文长度的关系"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解为什么用子词分词(词级/字符级的问题)。
   Understand why subword tokenization (problems of word/char level).
2. **从零实现 BPE 训练**(学合并规则)。
   Implement BPE training from scratch (learn merge rules).
3. **从零实现 BPE 编码**, 处理未登录词。
   Implement BPE encoding; handle unseen words.
4. 理解词表大小权衡与主流分词器。
   Understand vocab-size trade-offs and mainstream tokenizers.

## 目录 / TOC
1. [为什么要子词分词 ⭐](#1)
2. [BPE 训练：学合并规则（从零）⭐](#2)
3. [BPE 编码：切分新词（从零）⭐](#3)
4. [词表权衡与主流分词器 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么要子词分词 ⭐ / Why Subword Tokenization

切 token 有三种粒度，各有硬伤：
Three granularities, each with flaws:
- **词级(word)**：每个词一个 token。问题：①**词表爆炸**(英文几十万词，多语言更多)；②**未登录词(OOV)** ——没见过的词(新词、拼写变体、专有名词)无法表示；③ `run/running/runs` 当成毫不相关的三个 token。
  **Word-level:** one token per word. Problems: ① **vocab explosion** (hundreds of thousands); ② **OOV** — unseen words can't be represented; ③ `run/running/runs` treated as unrelated.
- **字符级(char)**：每个字符一个 token。问题：词表很小但**序列变得很长**(算力 ∝ 序列长²)、单个字符**语义太弱**。
  **Char-level:** one token per char. Tiny vocab but **very long sequences** (compute ∝ length²) and **weak per-token semantics**.
- **子词级(subword)**：折中——**常用词保留为整体，罕见词拆成有意义的片段**。例如 `tokenization → token + ization`，`unhappiness → un + happi + ness`。**没有 OOV**(最差也能拆到字符)，词表适中，语义合理。
  **Subword:** the compromise — **common words stay whole, rare words split into meaningful pieces**, e.g. `tokenization → token + ization`. **No OOV** (worst case down to chars), moderate vocab, sensible semantics.

几乎所有现代大模型(GPT、LLaMA…)都用子词。最经典的算法是 **BPE**。
Almost all modern LLMs (GPT, LLaMA…) use subwords. The classic algorithm is **BPE**.


<a id="2"></a>
## 2. BPE 训练：学合并规则（从零）⭐ / BPE Training From Scratch

**BPE(Byte-Pair Encoding)** 的训练极其简单(面试要能口述)：
**BPE training** is beautifully simple (be able to describe it):
1. 把每个词拆成**字符序列**(初始词表 = 所有字符)。
   Split each word into **characters** (initial vocab = all chars).
2. 统计语料里**所有相邻字符对**的出现频率。
   Count the frequency of **all adjacent pairs** in the corpus.
3. 把**最高频的那一对合并**成一个新 token，加入词表(记下这条合并规则)。
   **Merge the most frequent pair** into a new token, add to vocab (record the merge rule).
4. 重复 2–3 共 $N$ 次(合并次数决定最终词表大小)。
   Repeat 2–3 for $N$ times ($N$ controls final vocab size).

直觉：高频组合(如 `t`+`h`→`th`，`th`+`e`→`the`)会被逐步合并成整体；低频组合保持拆开。这样**常用模式变成单个 token，罕见词由片段拼成**。
Intuition: frequent combos (`t`+`h`→`th`, `th`+`e`→`the`) gradually merge into wholes; rare ones stay split. So **common patterns become single tokens; rare words are built from pieces**.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, re
from collections import Counter
import nltk; nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg
sns.set_theme(style="whitegrid")

text = gutenberg.raw("austen-sense.txt")[:100000].lower()
words = re.findall(r"[a-z]+", text)
word_freq = Counter(words)                                # 词频 / word frequencies
# 每个词表示为'字符元组', 末尾加 </w> 标记词尾 / each word as a tuple of chars + end marker
vocab = {tuple(list(w) + ["</w>"]): c for w, c in word_freq.items()}
print(f"语料 {len(words)} 词, {len(word_freq)} 个不同词")
print(f"初始: 每个词拆成字符, 如 'sister' → {tuple(list('sister')+['</w>'])}")

def get_pair_freqs(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        for i in range(len(word)-1):
            pairs[(word[i], word[i+1])] += freq           # 相邻对计数(按词频加权) / weighted adjacent-pair counts
    return pairs
def merge_pair(pair, vocab):
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = []; i = 0
        while i < len(word):
            if i < len(word)-1 and (word[i], word[i+1]) == pair:
                new_word.append(word[i] + word[i+1]); i += 2   # 合并这一对 / merge the pair
            else:
                new_word.append(word[i]); i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

merges = []
for step in range(200):                                   # 学 200 条合并规则 / learn 200 merges
    pairs = get_pair_freqs(vocab)
    if not pairs: break
    best = pairs.most_common(1)[0][0]                     # 最高频的相邻对 / most frequent pair
    vocab = merge_pair(best, vocab); merges.append(best)
print(f"\n前 15 条学到的合并规则(高频组合先被合并):")
print([a+b for a,b in merges[:15]])
print("观察: 先合并高频字母对(th,he,in,er...), 再合并成常见词/词缀(the, ing, and...)")


<a id="3"></a>
## 3. BPE 编码：切分新词（从零）⭐ / BPE Encoding From Scratch

学好合并规则后，**编码**一个词：从字符开始，**按学到的合并规则的顺序**反复合并，直到没有可合并的对。常见词会被合成**一个 token**，罕见/没见过的词会留成**几个子词片段**——这正是"无 OOV"的来源。
With merge rules learned, **encoding** a word: start from characters and repeatedly apply the **learned merges in order** until none apply. Common words collapse to **one token**; rare/unseen words remain **several subword pieces** — the source of "no OOV."


In [ ]:
merge_rank = {pair: i for i, pair in enumerate(merges)}   # 合并优先级(越早学的越先用) / merge priority

def bpe_encode(word):
    tokens = list(word) + ["</w>"]                        # 从字符开始 / start from chars
    while True:
        pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
        # 找当前可合并对里优先级最高的 / pick the highest-priority mergeable pair
        candidate = min((p for p in pairs if p in merge_rank), key=lambda p: merge_rank[p], default=None)
        if candidate is None: break                       # 没有可合并的了 / no more merges
        i = pairs.index(candidate)
        tokens = tokens[:i] + [tokens[i]+tokens[i+1]] + tokens[i+2:]
    return tokens

for w in ["the", "her", "sister", "happiness", "transformer", "antidisestablishment"]:
    toks = bpe_encode(w)
    seen = "高频词→整体(1 token)" if len(toks) == 1 else "较少见→拆成子词"
    print(f"  {w:22} → {toks}   ({len(toks)} tokens, {seen})")
print("\n最高频词(the/her)→1个token; 没那么高频的词(sister/happiness)→拆成几个子词; 绝不会OOV(最差拆到字符)")
print("(本例只学了200条合并规则; 真实GPT学几万条, 'sister'这类常用词也会成单个token)")


<a id="4"></a>
## 4. 词表权衡与主流分词器 + 小结 ⭐ / Vocab Trade-off & Mainstream Tokenizers

**合并次数 = 词表大小**，是个关键权衡(面试)：
**Number of merges = vocab size**, a key trade-off (interview):
- 词表**太小**(合并少)→ 接近字符级 → 序列**很长**(token 多)→ 算力/成本高、上下文装得少。
  **Too small** → near char-level → **long sequences** → costly, less fits in context.
- 词表**太大**(合并多)→ 接近词级 → 词表占内存、罕见 token 训练不充分。
  **Too large** → near word-level → vocab memory, undertrained rare tokens.
- 实际 GPT-2 用 **50257**、GPT-4 约 **10万**、多语言模型更大。
  Real GPT-2 uses **50,257**, GPT-4 ~**100k**, multilingual ones larger.

下面可视化：词表越大，编码同一段文本所需的 token 数越少(序列越短)。
Below: larger vocab → fewer tokens to encode the same text (shorter sequences).


In [ ]:
# 不同合并次数(词表大小)下, 编码一段测试文本需要多少 token / token count vs vocab size
test_words = re.findall(r"[a-z]+", gutenberg.raw("austen-emma.txt")[:5000].lower())
sizes = [0, 50, 100, 200]
counts = []
for n in sizes:
    mr = {p: i for i, p in enumerate(merges[:n])}
    def enc(word):
        tk = list(word)+["</w>"]
        while True:
            ps=[(tk[i],tk[i+1]) for i in range(len(tk)-1)]
            c=min((p for p in ps if p in mr), key=lambda p:mr[p], default=None)
            if c is None: break
            i=ps.index(c); tk=tk[:i]+[tk[i]+tk[i+1]]+tk[i+2:]
        return tk
    counts.append(sum(len(enc(w)) for w in test_words))
fig, ax = plt.subplots(figsize=(7,4))
ax.plot([s+len(set("".join(words))) for s in sizes], counts, "o-")   # x≈词表大小(字符+合并数) / vocab size
ax.set_xlabel("词表大小 (字符数 + 合并次数)"); ax.set_ylabel("编码测试文本所需 token 数")
ax.set_title("词表越大 → 同样文本所需 token 越少(序列越短) ← BPE 的核心权衡")
plt.tight_layout(); plt.show()
print(f"词表大小 vs token 数: {list(zip([s for s in sizes], counts))}")
print("更大词表→更短序列(省算力/省API成本/装更多上下文), 但词表本身更大; 实际取数万")
print("\n主流分词器: BPE(GPT系) / WordPiece(BERT, 按似然合并) / SentencePiece(直接对原始文本, 支持多语言无需预分词) / tiktoken(OpenAI高效BPE)")


```
为什么子词: 词级(词表爆炸+OOV) / 字符级(序列太长+语义弱); 子词折中=常见词整体+罕见词拆片+无OOV
BPE训练: 词拆成字符→统计相邻对频率→合并最高频对→重复N次(N=词表大小); 记录合并规则
BPE编码: 从字符开始按合并规则顺序反复合并; 常见词→1token, 罕见词→多子词
词表权衡: 太小→序列长(贵), 太大→词表占内存+罕见token欠训练; GPT-2=50257, GPT-4≈10万
token数=成本: API按token计费, 上下文窗口按token限长; 分词效率直接影响成本
分词器: BPE(GPT) / WordPiece(BERT) / SentencePiece(多语言/无需预分词) / tiktoken(OpenAI)
坑: 中文/代码/数字分词效率低(一个汉字可能多token); 分词影响算术/拼写等能力
```

### 💡 面试速查 / Interview cheat-sheet
1. **为什么子词**: 避免词表爆炸和OOV, 又比字符级短; 常见整体+罕见拆片。
   Why subword: avoid vocab explosion & OOV, shorter than char; common whole + rare split.
2. **BPE训练**: 反复合并最高频相邻对N次; N=词表大小。
   BPE training: repeatedly merge the most frequent pair N times; N = vocab size.
3. **BPE编码**: 按学到的合并规则顺序合并字符。
   BPE encoding: apply learned merges in order over characters.
4. **词表权衡**: 大→序列短但词表大; GPT-2=50257, GPT-4≈100k。
   Vocab trade-off: larger → shorter sequences but bigger vocab.
5. **分词器家族**: BPE/WordPiece/SentencePiece/tiktoken; token数=API成本+上下文长度。
   Tokenizers: BPE/WordPiece/SentencePiece/tiktoken; tokens = cost + context length.

### 下一节 / Next
**12.8 预训练 vs 微调**——大模型的两阶段范式: 先在海量文本上"预训练"学通用能力(极贵, 少数公司做), 再在特定任务上"微调"(便宜, 人人可做)。我们会梳理两者的概念、区别与何时用哪种。
**12.8 Pretraining vs Fine-tuning** — the two-stage LLM paradigm: "pretrain" on massive text for general ability (very expensive, few companies), then "fine-tune" on specific tasks (cheap, anyone). We'll clarify the concepts, differences, and when to use which.
